# 02 — Limpeza dos dados do ClinicalTrials.gov
Este notebook transforma os dados brutos (extração de 12/10/2026) em tabelas limpas para análise.


## 1. Ler os dados brutos

In [160]:
import json
import pandas as pd

In [161]:
with open("../data/raw/estudos_obesidade_2026-10-12.json", encoding="utf-8") as ficheiro:
    estudos_brutos = json.load(ficheiro)

In [162]:
len(estudos_brutos)

8431

In [163]:
print(json.dumps(estudos_brutos[0], indent=2, ensure_ascii=False))

{
  "protocolSection": {
    "identificationModule": {
      "nctId": "NCT06151145",
      "briefTitle": "Stepped Care for Weight Loss Maintenance"
    },
    "statusModule": {
      "overallStatus": "ACTIVE_NOT_RECRUITING",
      "startDateStruct": {
        "date": "2024-02-01",
        "type": "ACTUAL"
      },
      "primaryCompletionDateStruct": {
        "date": "2028-06",
        "type": "ESTIMATED"
      },
      "completionDateStruct": {
        "date": "2028-06",
        "type": "ESTIMATED"
      }
    },
    "sponsorCollaboratorsModule": {
      "leadSponsor": {
        "class": "OTHER"
      }
    },
    "designModule": {
      "studyType": "INTERVENTIONAL",
      "phases": [
        "NA"
      ],
      "enrollmentInfo": {
        "count": 192
      }
    },
    "armsInterventionsModule": {
      "interventions": [
        {
          "type": "BEHAVIORAL"
        },
        {
          "type": "BEHAVIORAL"
        },
        {
          "type": "BEHAVIORAL"
        }
      

In [164]:
print(json.dumps(estudos_brutos[0]["protocolSection"]["designModule"], indent=2))

{
  "studyType": "INTERVENTIONAL",
  "phases": [
    "NA"
  ],
  "enrollmentInfo": {
    "count": 192
  }
}


In [165]:
for estudo in estudos_brutos:
    status = estudo["protocolSection"]["statusModule"]
    if status["overallStatus"] == "TERMINATED":
        print(json.dumps(status, indent=2))
        break

{
  "overallStatus": "TERMINATED",
  "whyStopped": "difficulty in recruiting",
  "startDateStruct": {
    "date": "2013-12",
    "type": "ACTUAL"
  },
  "primaryCompletionDateStruct": {
    "date": "2022-01",
    "type": "ACTUAL"
  },
  "completionDateStruct": {
    "date": "2022-06",
    "type": "ACTUAL"
  }
}


## 2. Construir a tabela estudos

In [166]:
linhas = []
for estudo in estudos_brutos:
   
    protocolo = estudo.get("protocolSection", {})

    
    ident = protocolo.get("identificationModule", {})
    status =protocolo.get("statusModule",{})
    design = protocolo.get("designModule",{})
    patrocinio = protocolo.get("sponsorCollaboratorsModule",{})

    
    inicio = status.get("startDateStruct", {})
    fim_primaria = status.get("primaryCompletionDateStruct",{})
    fim = status.get("completionDateStruct",{})
    inscricao = design.get("enrollmentInfo",{})
    patrocinador = patrocinio.get("leadSponsor",{})

   
    fases = design.get("phases",[])

    linhas.append({
        "nct_id": ident.get("nctId"),
        "titulo": ident.get("briefTitle"),
        "tipo_estudo": design.get("studyType"),
        "estado": status.get("overallStatus"),
        "motivo_paragem": status.get("whyStopped"),
        "data_inicio": inicio.get("date"),
        "tipo_data_inicio": inicio.get("type"),
        "data_fim_primaria": fim_primaria.get("date"),
        "tipo_data_fim_primaria": fim_primaria.get("type"),
        "data_fim": fim.get("date"),
        "tipo_data_fim": fim.get("type"),
        "participantes": inscricao.get("count"),
        "fase": "/".join(fases) if fases else None,
        "classe_patrocinador": patrocinador.get("class"),
    })

df_estudos = pd.DataFrame(linhas)

In [167]:
df_estudos.shape

(8431, 14)

In [168]:
df_estudos.head()

,nct_id,titulo,tipo_estudo,estado,motivo_paragem,data_inicio,tipo_data_inicio,data_fim_primaria,tipo_data_fim_primaria,data_fim,tipo_data_fim,participantes,fase,classe_patrocinador
0,NCT06151145,Stepped Care for Weight Loss Maintenance,INTERVENTIONAL,ACTIVE_NOT_RECRUITING,NaN,2024-02-01,ACTUAL,2028-06,ESTIMATED,2028-06,ESTIMATED,192.0,NA,OTHER
1,NCT02185872,CrossFit Exercise to Improve Glucose Control f...,INTERVENTIONAL,COMPLETED,NaN,2012-02,NaN,2012-05,ACTUAL,2012-05,ACTUAL,23.0,NA,OTHER
2,NCT03917901,Computer-Delivered Intervention for Individual...,INTERVENTIONAL,COMPLETED,NaN,2019-09-16,ACTUAL,2021-03-06,ACTUAL,2021-03-06,ACTUAL,131.0,NA,OTHER
3,NCT01049698,Improving Muscle for Functional Independence T...,INTERVENTIONAL,COMPLETED,NaN,2010-02,NaN,2014-05,ACTUAL,2014-05,ACTUAL,126.0,NA,OTHER
4,NCT06989177,Probiotic Intervention on Body Weight,INTERVENTIONAL,RECRUITING,NaN,2025-06-10,ACTUAL,2026-12-30,ESTIMATED,2026-12-31,ESTIMATED,140.0,NA,OTHER_GOV


In [169]:
df_estudos.isna().sum()

nct_id                       0
titulo                       0
tipo_estudo                  0
estado                       0
motivo_paragem            7935
data_inicio                  0
tipo_data_inicio          2062
data_fim_primaria            3
tipo_data_fim_primaria       4
data_fim                    54
tipo_data_fim               57
participantes                2
fase                         3
classe_patrocinador          0
dtype: int64

In [170]:
parados = df_estudos["estado"].isin(["TERMINATED", "SUSPENDED", "WITHDRAWN"])
df_estudos[parados]["motivo_paragem"].isna().sum()

np.int64(40)

In [171]:
df_estudos["nct_id"].is_unique


True

In [172]:
df_estudos["tipo_estudo"].value_counts(dropna=False)

tipo_estudo
INTERVENTIONAL    8431
Name: count, dtype: int64

In [173]:
sem_tipo = df_estudos["tipo_data_inicio"].isna()
df_estudos[sem_tipo]["data_inicio"].str[:4].value_counts().sort_index()

data_inicio
2010    325
2011    317
2012    353
2013    323
2014    280
2015    263
2016    193
2017      8
Name: count, dtype: int64

In [174]:
df_estudos["tipo_data_inicio"].isna().mean()

np.float64(0.24457359743802634)

In [175]:
parados.sum()

np.int64(536)

In [176]:
df_estudos[~parados]["motivo_paragem"].notna().sum()

np.int64(0)

In [177]:
(df_estudos["data_fim"].notna() & df_estudos["tipo_data_fim"].isna()).sum()

np.int64(3)

In [178]:
df_estudos["participantes"]= df_estudos["participantes"].astype("Int64")

In [179]:
df_estudos.info()

<class 'pandas.DataFrame'>
RangeIndex: 8431 entries, 0 to 8430
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   nct_id                  8431 non-null   str  
 1   titulo                  8431 non-null   str  
 2   tipo_estudo             8431 non-null   str  
 3   estado                  8431 non-null   str  
 4   motivo_paragem          496 non-null    str  
 5   data_inicio             8431 non-null   str  
 6   tipo_data_inicio        6369 non-null   str  
 7   data_fim_primaria       8428 non-null   str  
 8   tipo_data_fim_primaria  8427 non-null   str  
 9   data_fim                8377 non-null   str  
 10  tipo_data_fim           8374 non-null   str  
 11  participantes           8429 non-null   Int64
 12  fase                    8428 non-null   str  
 13  classe_patrocinador     8431 non-null   str  
dtypes: Int64(1), str(13)
memory usage: 930.5 KB


### Observações da inspeção

- **Dimensão:** a tabela tem 8 431 estudos e 14 colunas.
- **Validações:** o `nct_id` é único (não há estudos repetidos) e os 8 431 estudos são todos INTERVENTIONAL, o que confirma o filtro da extração.
- **Estudos parados sem motivo:** dos 536 estudos parados, 40 não indicam o motivo. Não há estudos não parados com motivo.
- **Tipo da data de início:** 24,5 % dos estudos não têm tipo (real ou estimada). Todos começaram entre 2010 e 2017; a partir de 2018 não há nenhum caso, o que sugere uma mudança na forma como os dados eram recolhidos.
- **Data de fim:** 54 estudos não têm data de fim. Outros 3 têm a data, mas não indicam se é real ou estimada.
- **Data de fim primária:** 3 estudos não têm esta data. Outro 1 tem a data, mas não indica se é real ou estimada.
- **Participantes e fase:** 2 estudos não indicam o número de participantes e 3 não indicam a fase.

## 3. Construir a tabela estudo_pais


In [180]:
linhas_pais = []
for estudo in estudos_brutos:                      
    protocolo = estudo.get("protocolSection", {})
    ident = protocolo.get("identificationModule", {})
    locais = protocolo.get("contactsLocationsModule", {})
    for centro in locais.get("locations", []):           
        linhas_pais.append({
            "nct_id": ident.get("nctId"),
            "pais": centro.get("country"),
        })

df_estudo_pais = pd.DataFrame(linhas_pais)
df_estudo_pais.shape

(29961, 2)

In [181]:
df_estudo_pais= df_estudo_pais.drop_duplicates().reset_index(drop=True)
df_estudo_pais.shape

(9270, 2)

In [182]:
df_estudo_pais["nct_id"].nunique()


7837

In [183]:
df_estudo_pais.isna().sum()

nct_id    0
pais      0
dtype: int64

In [184]:
df_estudo_pais["pais"].nunique()

107

In [185]:
print(df_estudo_pais["pais"].value_counts().sort_index().to_string())

pais
Albania                      1
Algeria                      1
Argentina                   55
Australia                  114
Austria                     46
Bangladesh                   4
Belgium                     84
Bolivia                      1
Bosnia and Herzegovina       1
Brazil                     211
Brunei                       1
Bulgaria                    23
Cameroon                     1
Canada                     407
Chile                       42
China                      419
Colombia                    20
Costa Rica                   1
Croatia                     14
Cuba                         2
Cyprus                       9
Czechia                     73
Denmark                    228
Ecuador                      2
Egypt                      161
Estonia                      3
Ethiopia                     2
Finland                     68
France                     341
Georgia                      6
Germany                    213
Greece                      72
Gre

In [186]:
total_estudos = df_estudos["nct_id"].nunique()
estudos_com_pais = df_estudo_pais["nct_id"].nunique()
estudos_sem_pais = total_estudos - estudos_com_pais
round(estudos_sem_pais / total_estudos * 100, 1)

7.0

In [187]:
   sem_pais = ~df_estudos["nct_id"].isin(df_estudo_pais["nct_id"])
   print(sem_pais.sum())
   print(round(sem_pais.mean() * 100, 1))

594
7.0


In [188]:
df_estudo_pais["nct_id"].isin(df_estudos["nct_id"]).all()

np.True_

In [189]:
(df_estudo_pais["pais"] == "Portugal").sum()

np.int64(53)

### Observações da tabela estudo_pais

- **Dimensão:** 29 961 linhas (centros) antes de remover duplicados;9 270 pares estudo–país depois.
- **Estudos sem país:** 594 estudos (7 %) não indicam nenhum país.
- **Países:** 107 países diferentes. Não foram encontradas variantes de escrita.
- **Decisões:** 1 estudo indica "Serbia and Montenegro", nome anterior a 2006; mantido tal como registado. Os territórios (ex.: Puerto Rico, Hong Kong) foram mantidos separados, tal como no ClinicalTrials.gov.
- **Integridade:** todos os `nct_id` desta tabela existem na tabela `estudos`.
- **Portugal:** Participa em 53 estudos.

## 4. Construir a tabela estudo_intervencao

In [190]:
linhas_intervencao = []
for estudo in estudos_brutos:                      
    protocolo = estudo.get("protocolSection", {})
    ident = protocolo.get("identificationModule", {})
    modulo_interv = protocolo.get("armsInterventionsModule", {})
    for intervencao in modulo_interv.get("interventions", []):           
        linhas_intervencao.append({
            "nct_id": ident.get("nctId"),
            "tipo_intervencao": intervencao.get("type"),
        })

df_estudo_intervencao = pd.DataFrame(linhas_intervencao)
df_estudo_intervencao.shape

(16524, 2)

In [192]:
df_estudo_intervencao.isna().sum()

nct_id              0
tipo_intervencao    0
dtype: int64

In [193]:
df_estudo_intervencao["tipo_intervencao"].value_counts(dropna=False)

tipo_intervencao
BEHAVIORAL             5172
DRUG                   3607
OTHER                  3588
DIETARY_SUPPLEMENT     1974
DEVICE                  945
PROCEDURE               882
DIAGNOSTIC_TEST         150
BIOLOGICAL              110
COMBINATION_PRODUCT      66
GENETIC                  16
RADIATION                14
Name: count, dtype: int64

In [194]:
df_estudo_intervencao = df_estudo_intervencao.drop_duplicates().reset_index(drop=True)

In [195]:
type(df_estudo_intervencao)

pandas.DataFrame

In [198]:

df_estudo_intervencao["nct_id"].isin(df_estudos["nct_id"]).all()

np.True_

In [199]:
df_estudo_intervencao.shape

(9697, 2)

In [200]:
df_estudo_intervencao["tipo_intervencao"].value_counts()

tipo_intervencao
BEHAVIORAL             3107
OTHER                  2173
DRUG                   1790
DIETARY_SUPPLEMENT     1141
DEVICE                  683
PROCEDURE               590
DIAGNOSTIC_TEST          75
BIOLOGICAL               63
COMBINATION_PRODUCT      52
GENETIC                  13
RADIATION                10
Name: count, dtype: int64

In [202]:
sem_intervencao= ~df_estudos["nct_id"].isin(df_estudo_intervencao["nct_id"])


In [205]:
print(sem_intervencao.sum())
print(round(sem_intervencao.mean()*100,1))

0
0.0


In [206]:
df_estudo_intervencao["tipo_intervencao"].nunique() 

11

### Observações da tabela estudo_intervencao
- **Dimensão:** 16 524 linhas (intervenções) antes de remover duplicados; 9697 pares estudo–tipo depois.
- **Duplicados:** a maior descida foi no tipo DRUG (−50 %), o que é compatível com ensaios que registam o medicamento e o placebo, ambos como DRUG.
- **Estudos sem intervenção:** 0 (todos os estudos intervencionais indicam pelo menos uma intervenção).
- **Tipos:** 11 tipos, já padronizados pelo ClinicalTrials.gov; sem variantes de escrita.
- **Limitação:** o tipo OTHER (2 173 estudos) agrupa intervenções diversas e pouco específicas.
- **Nota para a análise:** um estudo pode ter vários tipos, por isso as contagens por tipo somam 9 697(mais do que os 8 431 estudos) e as percentagens por tipo não somam 100 %.
- **Integridade:** todos os `nct_id` desta tabela existem na tabela `estudos`.